## Library

In [4]:
import sys
import os 
import random

import torch 
import torch.nn as nn
from torch.optim import AdamW
from torchinfo import summary

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
from matplotlib import __version__ as matplotlib_version

from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn import __version__ as sklearn_version


## Version

In [10]:
print(f"Python version: {sys.version.split()[0]}")
print(f"PyTorch version: {torch.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Scikit-learn version: {sklearn_version}")
print(f"Matplotlib: {matplotlib_version}")
print(f"Pandas: {pd.__version__}")

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version (from torch): {torch.version.cuda}")
    print(f"cuDNN version: {torch.backends.cudnn.version()}")
    print(f"GPU device: {torch.cuda.get_device_name(0)}")

Python version: 3.13.2
PyTorch version: 2.8.0.dev20250409+cu128
NumPy version: 2.1.2
Scikit-learn version: 1.6.1
Matplotlib: 3.10.3
Pandas: 2.2.3
CUDA available: True
CUDA version (from torch): 12.8
cuDNN version: 90701
GPU device: NVIDIA GeForce RTX 3060 Laptop GPU


## Setting seed

In [12]:
def set_seed(seed=42):
    random.seed(seed)                        # Python random
    np.random.seed(seed)                     # NumPy random
    torch.manual_seed(seed)                  # PyTorch CPU
    torch.cuda.manual_seed(seed)             # PyTorch GPU
    torch.cuda.manual_seed_all(seed)         # PyTorch multi-GPU
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False  

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


## Data

In [30]:
df = load_breast_cancer()
X = df['data']
y = df['target']

print(X.shape,y.shape)

(569, 30) (569,)


## Data splitting

In [39]:
X_train,x_temp,y_train,y_temp = train_test_split(X,y,test_size=0.2,random_state=42)
X_val,X_test,y_val,y_test = train_test_split(x_temp,y_temp,test_size=0.5,random_state=42)

print(X_train.shape , y_train.shape)
print(X_val.shape , y_val.shape)
print(X_test.shape , y_test.shape)

(455, 30) (455,)
(57, 30) (57,)
(57, 30) (57,)


## Normalize

In [49]:
sc = StandardScaler()
X_train_sc = sc.fit_transform(X_train)
X_val_sc = sc.transform(X_val)
X_test_sc = sc.transform(X_test)

print(X_train.mean())
print(X_train_sc.mean())

61.72624167942857
-9.442670495742283e-16


## Data loader

In [124]:
class Dataset_cancer(torch.utils.data.Dataset):
    def __init__(self,X,y):
        self.X = torch.tensor(X.astype('float32'))
        self.y = torch.tensor(y.astype('float32')).view(-1,1)
        self.n_sample = self.X.size(0)

        print(self.X.size())
        print(self.y.size())
        print(self.X.dtype)
        print(self.y.dtype)
        print("*"*50)
      
    def __getitem__(self,idx):
        return self.X[idx] ,self.y[idx]
    
    def __len__(self):
        return self.n_sample


In [125]:
train_loader = torch.utils.data.DataLoader(Dataset_cancer(X_train_sc,y_train),batch_size=300,shuffle=True)
val_loader = torch.utils.data.DataLoader(Dataset_cancer(X_val_sc,y_val),batch_size=300,shuffle=False)
test_loader = torch.utils.data.DataLoader(Dataset_cancer(X_test_sc,y_test),batch_size=300,shuffle=False)

torch.Size([455, 30])
torch.Size([455, 1])
torch.float32
torch.float32
**************************************************
torch.Size([57, 30])
torch.Size([57, 1])
torch.float32
torch.float32
**************************************************
torch.Size([57, 30])
torch.Size([57, 1])
torch.float32
torch.float32
**************************************************


## Model 

In [ ]:
import pytorch_lightning as pl

In [171]:
class Model_classify(pl.LightningModule):
    def __init__(self, input_size , output_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size,output_size)
        self.criterion = nn.BCEWithLogitsLoss()

    def forward(self, X):
        return self.fc1(X)
    
    def training_step(self, batch,batch_idx):
        X , y = batch 
        logits = self(X)
        loss = self.criterion(logits,y)
        self.log("train_loss",loss,prog_bar=True)
        return loss  
    
    def validation_step(self, batch, batch_idx):
        X, y = batch
        logits = self(X)
        preds = (torch.sigmoid(logits) >= 0.5).long()
        acc = (preds == y.long()).float().mean()
        self.log("val_acc", acc, prog_bar=True)
    
    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=1e-3)
    

tensorboard --logdir=lightning_logs --port=6006


In [170]:
model = Model_classify(input_size=X_train_sc.shape[1],output_size=1)
trainer = pl.Trainer(max_epochs=10)
trainer.fit(model, train_loader, val_loader)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name      | Type              | Params | Mode 
--------------------------------------------------------
0 | fc1       | Linear            | 31     | train
1 | criterion | BCEWithLogitsLoss | 0      | train
--------------------------------------------------------
31        Trainable params
0         Non-trainable params
31        Total params
0.000     Total estimated model params size (MB)
2         Modules in train mode
0         Modules in eval mode


d:\mini\envs\pine\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
d:\mini\envs\pine\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
d:\mini\envs\pine\Lib\site-packages\pytorch_lightning\loops\fit_loop.py:310: The number of training batches (2) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


Epoch 9: 100%|██████████| 2/2 [00:00<00:00, 84.49it/s, v_num=1, train_loss=0.447, val_acc=0.912] 

`Trainer.fit` stopped: `max_epochs=10` reached.


Epoch 9: 100%|██████████| 2/2 [00:00<00:00, 59.04it/s, v_num=1, train_loss=0.447, val_acc=0.912]
